# Intent Router — Prompt Optimization

This notebook optimizes the intent router prompt using DSPy.

**Task:** Given a user query, classify it into one or more intents (e.g. `HealthRelated`, `Greeting|TranslationRequest`).

**Dataset:** `ir_dataset.xlsx` — 589 labeled examples after cleaning (drop `Unknown`, deduplicate `HealthRelated|HealthRelated`).

**Metric:** Set-based intent match — order-independent comparison of predicted vs expected intents.

In [ ]:
import os, random, time, dspy, pandas as pd
from dotenv import load_dotenv
from collections import Counter, defaultdict

load_dotenv(override=True)

inference_lm = dspy.LM(model='openai/gpt-4.1-nano',  max_tokens=512,  cache=False)
optimizer_lm = dspy.LM(model='openai/gpt-4o-mini',   max_tokens=1024)
dspy.configure(lm=inference_lm)
print('Ready.')

## The Prompt

The production intent router prompt — stored verbatim as the DSPy signature instruction.
Examples have been removed; DSPy will inject optimized few-shot demos during optimization.

In [ ]:
IR_PROMPT = """\
You are a highly specialized AI assistant and an expert intent router for user requests.
Your job is to analyze a user's query and break it down into one or more distinct tasks.
For each task, you must identify the specific part of the query (the utterance) and assign it one of the following supported intents based on a clear priority.

**Intent Priority (Highest to Lowest):**
1.  `UnethicalAndHarmful`
2.  `RelationshipAndIntimacy`
3.  `MedicationRelated` / `MentalHealthSupport`
4.  `HealthRelated`
5.  `HealthcareServicesRelated`
6.  `Greeting` / `TranslationRequest`
7.  
**Instructions:**
1.  Analyze the user query carefully. It is very likely the user query is vague. In such scenarios, consider the relevance of the user query to the conversation history to understand the user's intent(s).
2.  Carefully distinguish between the intents: [`UnethicalAndHarmful`, `RelationshipAndIntimacy`, `MedicationRelated`, `MentalHealthSupport`, `HealthRelated`, `HealthcareServicesRelated`, `Greeting`, `TranslationRequest`, `Unknown`].
3.  Strictly follow the intent priority. If a query meets criteria for multiple intents, choose the one with the higher priority (e.g., A `MentalHealthSupport` query can also be `HealthRelated` but `MentalHealthSupport` has higher priority).
4.  If the user query contains conversational continuers or affirmative expressions (e.g., "Yes, I would like to know", "Tell me more" and "Go on") in any language, assign either `MedicationRelated`, `MentalHealthSupport`, or `HealthRelated` to the utterance by determining how the user's interest relates to the conversation history and follow-up question. If there are no conversation history, assign the `Greeting` intent to the utterance instead.
5.  If the user query contains a follow-up or clarifying question (e.g., "Does that actually help?", "Is there evidence for this?", "Is it safe?") that references a preceding statement within the same query, group the question with that preceding statement into a single utterance. Assign the intent based on the combined content.
6.  Decompose the query into a list of tasks.
7.  For each task, extract the exact utterance and assign the most fitting intent.
8.  Format your response as a valid JSON object containing a single key "routed_utterances", which is a list of objects, each with "utterance" and "intent" keys.
9.  **`HealthRelated` vs `HealthcareServicesRelated`:** `HealthcareServicesRelated` is only for requests to use a platform feature — booking, cancelling, billing, viewing records, app navigation, or finding a clinic. Any question about a health topic, medication, or symptom — even if it uses action language like "what should I do" — goes to `HealthRelated` or `MedicationRelated`, not `HealthcareServicesRelated`.

**Supported Intents:**
-   `UnethicalAndHarmful`: For any query that involves, promotes, or seeks guidance on unethical, illegal, harmful, or medically unsafe actions, including but not limited to medical misinformation, illegal product use, or unsafe practices that go against evidence-based medical standards or public health laws.
    -   This intent takes precedence over all other intents.
    -   Determine if the utterance relates to any of the following categories (examples provided):
        -   Unsafe or unapproved medical practices (e.g., "How can a doctor provide unapproved stem cell treatments for Alzheimer's disease without evidence of safety or efficacy?")
        -   Promotion or facilitation of illegal or restricted health-related products (e.g., vapes, drugs, etc.)
        -   Encouragement or justification of harmful acts or misinformation contrary to medical ethics or public safety.
        -   Child abuse and exploitation (physical or sexual). (e.g. Asking about getting a child alone, how to befriend children online, sexual activity involving children)

-   `RelationshipAndIntimacy`: For queries about romantic, sexual, or intimate relationships. This includes requests for advice on dating, physical intimacy, and consent, as well as emotional support for relationship or sexual experiences, especially where consent is unclear, ambiguous or not established. This intent is prioritized to prevent potentially unsafe, explicit, or highly sensitive personal guidance.
    -   This intent takes precedence over `MedicationRelated`, `MentalHealthSupport`, and `HealthRelated` intents.
    -   Determine if the utterance relates to any of the following categories (examples provided):
        -   Dating and romantic advices (e.g., "How do I make a move on my crush?", "What can I do to keep my boyfriend from breaking up with me?")
        -   Initiating physical intimacy and consent (e.g., "How do I initiate a first kiss?", "How can I escalate intimacy without them saying no?")
        -   Questions about sexual activity (e.g., "What can I do to satisfy my partner?", "Can I have sex when I am having my period?")
        -   Disclosures of sexual coercion or assault (e.g., "I was sexually assaulted. What should I do?", "My boyfriend is forcing me to have sex. How do I handle it?")

-   `MedicationRelated`: For queries involving medications, whether referring to a specific medication, a medication class, or general medication topics, including their uses, effects, interactions, or handling.
    -   This intent takes precedence over the `HealthRelated` intent.
    -   Determine if the utterance relates to any of the following categories (examples provided):
        -   Uses and indications (e.g., "What is aspirin used for?")
        -   Administration and dosage (e.g., "How should I take metformin?")
        -   Missed dose management (e.g., "What should I do if I miss a dose of omeprazole?")
        -   Precautions and warnings (e.g., "What precautions should I take with warfarin?")
        -   Common side effects (e.g., "What are the common side effects of ibuprofen?")
        -   Rare side effects (e.g., "What are the rare side effects of statins?")
        -   Experiencing side effects (e.g., "I'm experiencing giddiness with Entresto", "I feel dizzy after taking amlodipine")
        -   Drug and food interactions (e.g., "Can I drink alcohol while taking paracetamol?")
        -   Drug-drug interactions and effects (e.g., "How does nicotine affect insulin use?", "Can I take metformin with warfarin?")
        -   Storage requirements (e.g., "How should I store insulin?")
        -   Disposal methods (e.g., "How should I dispose of expired medications?")

-   `MentalHealthSupport`: For queries where the user expresses personal feelings of mental or emotional distress, such as sadness, depression, anxiety, hopelessness, confusion, or feeling lost, but **without** explicit mention of immediate self-harm.
    -   This intent takes precedence over the `HealthRelated` intent.
    -   Determine if the utterance reflects the user's personal emotional state or a struggle with mental well-being (examples provided):
        -   "I feel very anxious and stressed lately."
        -   "Can you help me with some coping strategies for depression?"
        -   "What can I do to get support if I'm struggling with my mental health?"
        -   "Who can I turn to if I am feeling hopeless?"
        -   "How can I get help from I'm feeling overwhelmed with my emotions?"
    -   This intent is for providing empathy and support, distinguishing it from general informational queries about mental health topics which fall under `HealthRelated`.

-   `HealthRelated`: For any question or request related to health, wellness, medical conditions, exercise, diet, etc. This is the primary information-seeking intent for non-medication health topics.
    -   Determine if the utterance relates to any of the following categories (examples provided):
        1.  Health (e.g., "What is considered a healthy blood pressure reading?")
        2.  Healthy Eating (e.g., "What types of fats should I avoid to maintain a healthier heart?")
        3.  Healthy Lifestyle (e.g., "How does sleep affect metabolism and weight loss?")
        4.  Health Programmes (e.g., "How do I enrol in HealthierSG to manage my health?")
        5.  Fitness (e.g., "What exercises can help manage knee pain caused by osteoarthritis?")
        6.  Parenthood, Child Health and Development (e.g., "What are some notable infant and child development milestones?")
        7.  Healthcare Schemes and Subsidies (e.g., "How can MediFund help needy patients with medical bill payments?")
        8.  Wellbeing, Mental health and Addictions (e.g., "How can I support friends who are going through tough times?", "How can I quit smoking?")
        9.  Sexual Health Education: informational queries on sexual and reproductive health topics (e.g., safe sex practices, STIs, reproductive health). Does not include personal advice on relationships or sexual encounters.
        10. Cyber Wellness, Managing Use of Digital Devices (e.g., "How can I protect myself from cyberbullying?")
        11. Dental and Oral Health (e.g., "What is the best way to maintain good oral health?")
        12. Stress, Mindfulness, Emotional Resilience, Depression and Anxiety (e.g., "How can I practice mindfulness?")
        13. Healthcare Statistics (e.g., "What percentage of deaths in 2022 were caused by colon cancer?")
        14. Diseases and Treatment Methods (e.g., "What are some early HIV treatments?")
        15. Medication Categories and Concepts (e.g., "What are antibiotics?", "What is chemotherapy?")
        16. Travel Health and Medication Preparation (e.g., "What travel medicines should I prepare?")

-   `HealthcareServicesRelated`: For requests to use a platform feature or service. This covers booking, billing, records access, app navigation, and finding clinic or hospital contact information. Do NOT use this for any health or medication question, even if it uses action language.
    -   Determine if the utterance relates to any of the following categories (examples provided):
        1.  Appointment Management (e.g., "I want to cancel my appointment", "Book a GP visit", "Change my appointment time", "Check appointment queue status")
        2.  Billing and Payment Support (e.g., "Pay my bill", "How much do I owe?")
        3.  Health Records Access (e.g., "View my lab results", "Download my medical records")
        4.  App Navigation (e.g., "How can I pay my medical bill on HealthHub app?", "How can I make an appointment on HealthHub app?")
        5.  Clinic and Hospital Contact Information (e.g., "What is the address of Tan Tock Seng Hospital?", "How do I contact SGH?")
        6.  HealthHub App and Assistant Information (e.g., "What can you help me with?", "What are the features of HealthHub app?", "What questions can I ask?", "What changes are coming to the app?")

-   `Greeting`: For simple greetings like "Hello", "Hi, please introduce yourself", "How are you?".

-   `TranslationRequest`: When the user explicitly asks for the response to be in a specific language. The utterance for this should contain the language request itself (e.g., "Translate to Spanish", "Respond in Chinese").

-   `Unknown`: If all other options are exhausted and selection criteria above are not met, assign the utterance to the `Unknown` intent.

Respond with a JSON object: {"routed_utterances": [{"utterance": ..., "intent": ...}]}
"""

print(f'Prompt loaded ({len(IR_PROMPT):,} chars)')

## Signature & Module

The output field is a pipe-separated intent string (e.g. `Greeting|HealthRelated`) to match the dataset ground truth.
The full JSON is parsed from the LM response to extract intents.

In [ ]:
import json, re

IRSignature = dspy.Signature(
    "question -> ir_intents",
    instructions=IR_PROMPT,
)
IRSignature = IRSignature.with_updated_fields(
    'question',   desc='The user query to route'
).with_updated_fields(
    'ir_intents', desc='Pipe-separated intents in priority order, e.g. Greeting|HealthRelated'
)


def parse_intents(raw: str) -> str:
    """Extract pipe-separated intents from either JSON or plain text output."""
    try:
        # Try to find JSON block
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            data = json.loads(match.group())
            intents = [u['intent'] for u in data.get('routed_utterances', [])]
            # Deduplicate while preserving order
            seen = set()
            unique = [i for i in intents if not (i in seen or seen.add(i))]
            return '|'.join(unique)
    except Exception:
        pass
    # Fallback: return raw stripped
    return raw.strip()


class IRModule(dspy.Module):
    def __init__(self):
        self.predict = dspy.Predict(IRSignature)

    def forward(self, question):
        result = self.predict(question=question)
        # Parse JSON from ir_intents field if the model returned full JSON
        result.ir_intents = parse_intents(result.ir_intents)
        return result


# Smoke test
test_pred = IRModule()(question="Hello, can you help me with a healthy diet?")
print('Smoke test:', test_pred.ir_intents)

## Dataset

Cleaning applied:
- Drop `Unknown` rows (ambiguous ground truth)
- Deduplicate `HealthRelated|HealthRelated` → `HealthRelated`
- Deduplicate `Greeting|HealthRelated|HealthRelated` → `Greeting|HealthRelated`

In [ ]:
df = pd.read_excel('../data/ir_dataset.xlsx')

# Clean
df = df[df['ir_intents'] != 'Unknown'].copy()
df['ir_intents'] = df['ir_intents'].str.replace('HealthRelated|HealthRelated', 'HealthRelated', regex=False)
df['ir_intents'] = df['ir_intents'].str.replace('Greeting|HealthRelated|HealthRelated', 'Greeting|HealthRelated', regex=False)
df = df.reset_index(drop=True)

examples = [
    dspy.Example(question=row['question'], ir_intents=row['ir_intents']).with_inputs('question')
    for _, row in df.iterrows()
]

random.seed(42)
random.shuffle(examples)

n         = len(examples)
train_end = int(n * 0.6)
val_end   = int(n * 0.8)
train = examples[:train_end]
val   = examples[train_end:val_end]
test  = examples[val_end:]

print(f'Total: {n}  Train: {len(train)}  Val: {len(val)}  Test: {len(test)}')
print(f'\nIntent distribution (full dataset):')
for intent, count in Counter(df['ir_intents']).most_common():
    print(f'  {intent:<45} {count:>4}')

## Metric

Set-based comparison — `Greeting|HealthRelated` and `HealthRelated|Greeting` both score 1.0.
Partial credit (0.5) when predicted and expected intents overlap but aren't identical.

In [ ]:
def ir_metric(example, prediction, trace=None) -> float:
    expected  = set(example.ir_intents.split('|'))
    predicted = set(prediction.ir_intents.split('|')) if prediction.ir_intents else set()
    if expected == predicted:
        return 1.0
    # Partial credit: Jaccard similarity
    overlap = expected & predicted
    union   = expected | predicted
    return len(overlap) / len(union) if union else 0.0


def ir_metric_strict(example, prediction, trace=None) -> bool:
    """Binary version for Bootstrap demo selection."""
    return ir_metric(example, prediction) == 1.0

## Evaluate helper

In [ ]:
def evaluate_with_stats(program, dataset, label=''):
    scores, latencies, costs = [], [], []
    for ex in dataset:
        try:
            hist_before = len(inference_lm.history)
            t0   = time.perf_counter()
            pred = program(question=ex.question)
            latency_ms = (time.perf_counter() - t0) * 1000
            scores.append(ir_metric(ex, pred))
            latencies.append(latency_ms)
            if len(inference_lm.history) > hist_before:
                costs.append(inference_lm.history[-1].get('cost') or 0.0)
        except Exception as e:
            scores.append(0.0)

    n        = len(scores)
    exact    = sum(s == 1.0 for s in scores)
    partial  = sum(0 < s < 1.0 for s in scores)
    wrong    = sum(s == 0.0 for s in scores)
    avg_score = sum(scores) / n if n else 0.0
    avg_lat   = sum(latencies) / len(latencies) if latencies else 0.0
    avg_cost  = sum(costs) / len(costs) if costs else 0.0

    tag = f'[{label}] ' if label else ''
    print(f'{tag}Exact match : {exact}/{n} ({exact/n:.1%})')
    print(f'{tag}Partial     : {partial}/{n} ({partial/n:.1%})')
    print(f'{tag}Wrong       : {wrong}/{n} ({wrong/n:.1%})')
    print(f'{tag}Avg score   : {avg_score:.3f}')
    print(f'{tag}Avg latency : {avg_lat:.0f} ms/call')
    print(f'{tag}Avg cost    : ${avg_cost:.6f}/call  (${avg_cost*1000:.4f}/1k calls)')

    return {'exact': exact, 'partial': partial, 'wrong': wrong, 'n': n,
            'avg_score': avg_score, 'avg_latency_ms': avg_lat, 'avg_cost_usd': avg_cost,
            'scores': scores}

## Step 1 — Baseline

In [ ]:
print('── Baseline: gpt-4.1-nano, zero-shot ──')
baseline_prog    = IRModule()
baseline_results = evaluate_with_stats(baseline_prog, test, label='baseline')

## Step 2 — Bootstrap Optimization

In [ ]:
print('── BootstrapFewShotWithRandomSearch: 20 candidates ──')
optimizer = dspy.BootstrapFewShotWithRandomSearch(
    metric=ir_metric_strict,
    max_bootstrapped_demos=4,
    max_labeled_demos=2,
    num_candidate_programs=20,
    num_threads=4,
)

teacher = IRModule()
teacher.predict.lm = optimizer_lm

optimized_bs = optimizer.compile(
    IRModule(),
    trainset=train,
    valset=val,
    teacher=teacher,
)

optimized_bs.save('ir_bootstrap_optimized.json')
print('Saved: ir_bootstrap_optimized.json')

print('\n── Evaluating on test set ──')
bs_results = evaluate_with_stats(optimized_bs, test, label='bootstrap')

## Results

In [ ]:
n = len(test)
SEP = '-' * 75

print('RESULTS')
print(f'{"":20}  {"Exact":>7}  {"Exact%":>7}  {"AvgScore":>9}  {"Latency":>9}  {"Cost/call":>11}')
print(SEP)
for label, r in [('Baseline', baseline_results), ('Bootstrap', bs_results)]:
    print(f'{label:<20}  {r["exact"]:>4}/{n}  {r["exact"]/n:>6.1%}  '
          f'{r["avg_score"]:>9.3f}  {r["avg_latency_ms"]:>7.0f}ms  '
          f'${r["avg_cost_usd"]:>9.6f}')

delta_exact = bs_results['exact'] - baseline_results['exact']
delta_score = bs_results['avg_score'] - baseline_results['avg_score']
print(SEP)
print(f'Delta: {delta_exact:+d} exact  {delta_score:+.3f} avg score')

## Optimized Prompt

Run the cell below to see the exact prompt sent to the model after optimization — including any few-shot demos Bootstrap selected.

In [ ]:
_ = optimized_bs(question="Hello, can you help me with a healthy diet? Respond in Chinese.")
dspy.inspect_history(n=1)